# Tutorial: 2D Poisson with `volumential` + `pytential`

This notebook is a practical tutorial for new `volumential` users. We solve a smooth Poisson problem on a nontrivial starfish domain using:

- a volume potential on box-quadrature points,
- harmonic continuation of source values outside the physical domain,
- a harmonic boundary correction to enforce Dirichlet data.

Because we use a manufactured exact solution, every stage can be validated with concrete error metrics.



## Roadmap

What this tutorial covers:

1. **Problem setup**: geometry, exact solution, and forcing term.
2. **One-time infrastructure**: OpenCL context and near-field table cache.
3. **Reusable solver**: a single `run_poisson_case(...)` driver.
4. **Single run diagnostics**: inspect geometry, source extension behavior, and error maps.
5. **Uniform co-refinement**: refine volume + boundary + QBX together and measure convergence.
6. **Result interpretation**: summarize convergence and solver health.
7. **Boundary-focused AMR**: refine only boundary-intersecting volume boxes and compare cost/accuracy.

Tip: this notebook is designed so you can copy `run_poisson_case(...)` into your own project and swap in your geometry and boundary data.



In [1]:
from __future__ import annotations

from functools import partial
import numpy as np
from scipy.spatial import cKDTree

import pyopencl as cl
import pyopencl.array

from arraycontext import flatten, unflatten
from meshmode.dof_array import DOFArray
from meshmode.discretization import Discretization
from meshmode.discretization.poly_element import InterpolatoryQuadratureSimplexGroupFactory
from meshmode.mesh.generation import make_curve_mesh, starfish

from boxtree.array_context import PyOpenCLArrayContext as BoxtreePyOpenCLArrayContext
from boxtree.traversal import FMMTraversalBuilder

from pytools.obj_array import new_1d as obj_array_1d

from pytential.array_context import PyOpenCLArrayContext
from pytential.qbx import QBXLayerPotentialSource
from pytential.target import PointsTarget

import volumential.meshgen as mg
from volumential.expansion_wrangler_fpnd import (
    FPNDExpansionWrangler,
    FPNDTreeIndependentDataForWrangler,
)
from volumential.function_extension import compute_harmonic_extension
from volumential.nearfield_potential_table import DuffyBuildConfig
from volumential.table_manager import NearFieldInteractionTableManager
from volumential.tree_interactive_build import build_particle_tree_from_box_tree
from volumential.volume_fmm import drive_volume_fmm, interpolate_volume_potential

from sumpy.expansion import DefaultExpansionFactory
from sumpy.kernel import LaplaceKernel

/home/xywei/Work/fmm/volumential-ipa-run/volumential/__init__.py:41: UserWarning: pytools.persistent_dict 'volumential-code-cache-v0-2017.1a0': enabling safe_sync as default. This provides strong protection against data loss, but can be unnecessarily expensive for use cases such as caches.Pass 'safe_sync=False' if occasional data loss is tolerable. Pass 'safe_sync=True' to suppress this warning.
  code_cache = WriteOncePersistentDict("volumential-code-cache-v0-"+VERSION_TEXT)


## 1) Problem Definition

We build a manufactured-solution test case so the true error is known everywhere.

- Domain boundary: `starfish(t)` from `meshmode`.
- Exact solution: smooth oscillatory modes plus a Gaussian bump.
- Forcing: `f = -laplacian(u_exact)`.
- Evaluation set: fixed interior probe points inside the physical starfish domain (`probe_scale=1.00` by default) so boundary-region error is reflected in reported metrics.



In [ ]:
from matplotlib.path import Path

dim = 2
bbox_a, bbox_b = -1.30, 1.30
q_order = 4

table_filename = 'nft_poisson2d_starfish_demo.sqlite'
table_root_extent = bbox_b - bbox_a

bdry_order = 4
gmres_tolerance = 1.0e-8


def u_exact(x, y):
    phase1 = 3.0 * x + 2.0 * y
    phase2 = 4.0 * x - 1.5 * y
    phase3 = 2.5 * x - 3.5 * y

    smooth_wave = 0.55 * np.sin(phase1) + 0.35 * np.cos(phase2)
    modulated_wave = 0.25 * np.exp(0.4 * x - 0.3 * y) * np.sin(phase3)

    bump_r2 = (x - 0.25) ** 2 + (y + 0.15) ** 2
    bump = 0.30 * np.exp(-5.0 * bump_r2)

    return smooth_wave + modulated_wave + bump


def laplacian_u_exact(x, y):
    phase1 = 3.0 * x + 2.0 * y
    phase2 = 4.0 * x - 1.5 * y

    term1 = -0.55 * (3.0**2 + 2.0**2) * np.sin(phase1)
    term2 = -0.35 * (4.0**2 + 1.5**2) * np.cos(phase2)

    a, b = 0.4, -0.3
    c, d = 2.5, -3.5
    phase3 = c * x + d * y
    envelope = np.exp(a * x + b * y)
    term3 = 0.25 * envelope * (
        (a * a + b * b - c * c - d * d) * np.sin(phase3)
        + 2.0 * (a * c + b * d) * np.cos(phase3)
    )

    sigma = 5.0
    bump_r2 = (x - 0.25) ** 2 + (y + 0.15) ** 2
    term4 = 0.30 * (4.0 * sigma * sigma * bump_r2 - 4.0 * sigma) * np.exp(-sigma * bump_r2)

    return term1 + term2 + term3 + term4


def rhs_f(x, y):
    return -laplacian_u_exact(x, y)


def make_starfish_path(scale=1.0, npts=4096):
    t = np.linspace(0.0, 1.0, npts, endpoint=False)
    curve = starfish(t)
    vertices = np.ascontiguousarray(np.vstack([scale * curve[0], scale * curve[1]]).T)
    closed_vertices = np.vstack([vertices, vertices[0]])
    return Path(closed_vertices), closed_vertices


starfish_path, starfish_vertices = make_starfish_path(scale=1.0, npts=4096)
probe_scale = 1.00
probe_path, _ = make_starfish_path(scale=probe_scale, npts=4096)

probe_axis = np.linspace(-1.15, 1.15, 65)
probe_xx, probe_yy = np.meshgrid(probe_axis, probe_axis, indexing='xy')
probe_candidates = np.ascontiguousarray(np.vstack([probe_xx.ravel(), probe_yy.ravel()]).T)
probe_mask = probe_path.contains_points(probe_candidates)
probe_points_host = np.ascontiguousarray(probe_candidates[probe_mask])

print(f'fixed interior probe points (starfish scale={probe_scale:.2f}):', probe_points_host.shape[0])


## 2) Infrastructure Setup

Create OpenCL/QBX contexts and build or load the near-field interaction table once.

This step is intentionally separate so repeated parameter studies can reuse the same cached table.



In [3]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx)
actx = PyOpenCLArrayContext(queue)

probe_target = PointsTarget(actx.freeze(actx.from_numpy(np.ascontiguousarray(probe_points_host.T))))

tm = NearFieldInteractionTableManager(
    table_filename,
    root_extent=table_root_extent,
    queue=queue,
)
build_config = DuffyBuildConfig(
    radial_rule='tanh-sinh-fast',
    regular_quad_order=8,
    radial_quad_order=21,
)

nftable, _ = tm.get_table(
    dim,
    'Laplace',
    q_order,
    force_recompute=False,
    queue=queue,
    build_config=build_config,
)

print('near-field table ready:', table_filename)


SYCL CPU RT Warning: Unknown host CPU.


near-field table ready: nft_poisson2d_starfish_demo.sqlite


## 3) Solver Module

`run_poisson_case(...)` executes one full Poisson solve and returns diagnostics.

Pipeline per run:

1. Build boundary discretization + QBX objects.
2. Build (uniform or boundary-refined) volume mesh and quadrature points.
3. Evaluate interior source values from `f` and harmonically continue exterior source values.
4. Compute volume potential with `volumential` FMM.
5. Solve harmonic boundary correction for `g - v|_boundary`.
6. Report manufactured-solution errors and GMRES states.

This is the function to reuse when you run your own parameter sweeps.



In [4]:
def build_boundary_qbx(nelements_bdry, qbx_order):
    bdry_mesh = make_curve_mesh(
        starfish,
        np.linspace(0.0, 1.0, nelements_bdry + 1),
        bdry_order,
    )

    density_discr = Discretization(
        actx,
        bdry_mesh,
        InterpolatoryQuadratureSimplexGroupFactory(bdry_order),
    )

    qbx = QBXLayerPotentialSource(
        density_discr,
        fine_order=4 * bdry_order,
        qbx_order=qbx_order,
        fmm_order=False,
    )

    bdry_nodes = actx.thaw(density_discr.nodes())

    flat_bdry_nodes = flatten(bdry_nodes, actx, leaf_class=DOFArray)
    bdry_x_host = actx.to_numpy(flat_bdry_nodes[0])
    bdry_y_host = actx.to_numpy(flat_bdry_nodes[1])
    bdry_points_host = np.ascontiguousarray(np.vstack([bdry_x_host, bdry_y_host]))

    g_bdry_flat_host = np.ascontiguousarray(u_exact(bdry_x_host, bdry_y_host))
    g_bdry_flat = actx.from_numpy(g_bdry_flat_host)
    g_bdry = unflatten(bdry_nodes[0], g_bdry_flat, actx)

    return qbx, density_discr, bdry_nodes, g_bdry, bdry_points_host


def refine_volume_mesh_near_boundary(
    volume_mesh,
    boundary_path,
    boundary_vertices,
    *,
    max_level=10,
    max_passes=32,
    near_boundary_factor=0.20,
):
    boundary_vertices = np.ascontiguousarray(boundary_vertices[:-1])
    boundary_kdtree = cKDTree(boundary_vertices)

    n_refined_total = 0
    n_passes = 0

    for _ in range(max_passes):
        leaf_boxes = volume_mesh._leaf_boxes()
        leaf_levels = volume_mesh._leaf_levels()

        if len(leaf_boxes) == 0:
            break

        eligible = leaf_levels < max_level
        if not np.any(eligible):
            break

        centers = volume_mesh._leaf_centers()
        sizes = volume_mesh._leaf_side_lengths()
        half = 0.5 * sizes

        x = centers[:, 0]
        y = centers[:, 1]

        corners = np.ascontiguousarray(
            np.vstack(
                [
                    np.column_stack([x - half, y - half]),
                    np.column_stack([x + half, y - half]),
                    np.column_stack([x + half, y + half]),
                    np.column_stack([x - half, y + half]),
                ]
            )
        )

        edge_midpoints = np.ascontiguousarray(
            np.vstack(
                [
                    np.column_stack([x, y - half]),
                    np.column_stack([x + half, y]),
                    np.column_stack([x, y + half]),
                    np.column_stack([x - half, y]),
                ]
            )
        )

        stencil_points = np.ascontiguousarray(
            np.vstack([corners, edge_midpoints, centers])
        )
        stencil_inside = boundary_path.contains_points(stencil_points).reshape(9, -1).T
        stencil_mixed = np.any(stencil_inside, axis=1) & (~np.all(stencil_inside, axis=1))

        dist_to_boundary, _ = boundary_kdtree.query(centers)
        near_boundary = dist_to_boundary <= (
            near_boundary_factor * np.sqrt(2.0) * sizes + 1.0e-14
        )

        refine_leaf_mask = eligible & (stencil_mixed | near_boundary)
        if not np.any(refine_leaf_mask):
            break

        refine_flags = np.zeros(volume_mesh.boxtree.nboxes, dtype=bool)
        refine_flags[leaf_boxes[refine_leaf_mask]] = True
        coarsen_flags = np.zeros_like(refine_flags)

        n_refined_total += int(np.count_nonzero(refine_leaf_mask))
        n_passes += 1

        volume_mesh.boxtree.refine_and_coarsen(
            refine_flags=refine_flags,
            coarsen_flags=coarsen_flags,
            error_on_ignored_flags=False,
        )

    leaf_levels = volume_mesh._leaf_levels()
    max_leaf_level = int(np.max(leaf_levels)) if len(leaf_levels) else -1

    return {
        'passes': int(n_passes),
        'n_refined_total': int(n_refined_total),
        'n_active_cells': int(volume_mesh.n_active_cells()),
        'max_leaf_level': max_leaf_level,
    }


def run_poisson_case(
    *,
    vol_nlevels,
    nelements_bdry,
    qbx_order,
    fmm_order,
    adaptive_boundary_max_level=None,
    adaptive_max_passes=32,
    adaptive_near_boundary_factor=0.20,
    return_fields=False,
    return_mesh=False,
):
    qbx, density_discr, bdry_nodes, g_bdry, bdry_points_host = build_boundary_qbx(
        nelements_bdry=nelements_bdry,
        qbx_order=qbx_order,
    )

    volume_mesh = mg.MeshGen2D(
        degree=q_order,
        nlevels=vol_nlevels,
        a=bbox_a,
        b=bbox_b,
        queue=queue,
    )

    if adaptive_boundary_max_level is not None and adaptive_boundary_max_level > vol_nlevels:
        mesh_info = refine_volume_mesh_near_boundary(
            volume_mesh,
            starfish_path,
            starfish_vertices,
            max_level=adaptive_boundary_max_level,
            max_passes=adaptive_max_passes,
            near_boundary_factor=adaptive_near_boundary_factor,
        )
    else:
        leaf_levels = volume_mesh._leaf_levels()
        mesh_info = {
            'passes': 0,
            'n_refined_total': 0,
            'n_active_cells': int(volume_mesh.n_active_cells()),
            'max_leaf_level': int(np.max(leaf_levels)) if len(leaf_levels) else -1,
        }

    source_points_host = np.ascontiguousarray(volume_mesh.get_q_points())
    source_weights_host = np.ascontiguousarray(volume_mesh.get_q_weights())

    rhs_full_host = rhs_f(source_points_host[:, 0], source_points_host[:, 1])
    interior_source_mask = starfish_path.contains_points(source_points_host)
    exterior_source_mask = ~interior_source_mask

    source_vals_host = np.ascontiguousarray(rhs_full_host.copy())

    rhs_bdry_flat_host = np.ascontiguousarray(rhs_f(bdry_points_host[0], bdry_points_host[1]))
    rhs_bdry = unflatten(
        bdry_nodes[0],
        actx.from_numpy(rhs_bdry_flat_host),
        actx,
    )

    gmres_rhs_ext_state = 'not-needed'
    if np.any(exterior_source_mask):
        exterior_source_points_host = np.ascontiguousarray(source_points_host[exterior_source_mask])
        exterior_source_target = PointsTarget(
            actx.freeze(actx.from_numpy(np.ascontiguousarray(exterior_source_points_host.T)))
        )

        rhs_exterior, dbg_rhs_ext = compute_harmonic_extension(
            queue,
            exterior_source_target,
            qbx,
            density_discr,
            rhs_bdry,
            loc_sign=+1,
            representation_mode='auto',
            target_association_tolerance=0.05,
            gmres_tolerance=gmres_tolerance,
            actx=actx,
        )
        source_vals_host[exterior_source_mask] = actx.to_numpy(rhs_exterior)
        gmres_rhs_ext_state = dbg_rhs_ext['gmres_result'].state

    all_targets_host = np.ascontiguousarray(
        np.hstack([probe_points_host.T, bdry_points_host])
    )

    target_points = obj_array_1d(
        [
            cl.array.to_device(queue, np.ascontiguousarray(all_targets_host[iaxis]))
            for iaxis in range(dim)
        ]
    )

    source_vals = cl.array.to_device(queue, np.ascontiguousarray(source_vals_host))
    source_weights = cl.array.to_device(queue, source_weights_host)

    bt_actx = BoxtreePyOpenCLArrayContext(queue)
    source_tree = build_particle_tree_from_box_tree(
        bt_actx,
        volume_mesh.boxtree,
        source_points_host,
    )

    trav_builder = FMMTraversalBuilder(bt_actx)
    source_trav, _ = trav_builder(bt_actx, source_tree)

    knl = LaplaceKernel(dim)
    expn_factory = DefaultExpansionFactory()
    local_expn_class = expn_factory.get_local_expansion_class(knl)
    mpole_expn_class = expn_factory.get_multipole_expansion_class(knl)

    tree_indep = FPNDTreeIndependentDataForWrangler(
        ctx,
        partial(mpole_expn_class, knl),
        partial(local_expn_class, knl),
        [knl],
        exclude_self=False,
    )

    wrangler = FPNDExpansionWrangler(
        tree_indep=tree_indep,
        traversal=source_trav,
        near_field_table=nftable,
        dtype=np.float64,
        fmm_level_to_order=lambda kernel, kernel_args, tree, lev: fmm_order,
        quad_order=q_order,
        queue=queue,
    )

    (vf_source,) = drive_volume_fmm(
        source_trav,
        wrangler,
        source_vals * source_weights,
        source_vals,
    )

    vf_all = interpolate_volume_potential(
        target_points,
        source_trav,
        wrangler,
        vf_source,
    )
    vf_all_host = vf_all.get() if hasattr(vf_all, 'get') else np.asarray(vf_all)
    n_probe = probe_points_host.shape[0]

    vf_probe_host = vf_all_host[:n_probe]
    vf_bdry_host = vf_all_host[n_probe:]

    g_bdry_flat_host = np.ascontiguousarray(u_exact(bdry_points_host[0], bdry_points_host[1]))
    corr_bdry_flat = actx.from_numpy(np.ascontiguousarray(g_bdry_flat_host - vf_bdry_host))
    corr_bdry = unflatten(bdry_nodes[0], corr_bdry_flat, actx)

    corr_probe, dbg_corr = compute_harmonic_extension(
        queue,
        probe_target,
        qbx,
        density_discr,
        corr_bdry,
        loc_sign=-1,
        representation_mode='auto',
        target_association_tolerance=0.05,
        gmres_tolerance=gmres_tolerance,
        actx=actx,
    )
    corr_probe_host = actx.to_numpy(corr_probe)

    u_num_host = vf_probe_host + corr_probe_host
    u_ref_host = u_exact(probe_points_host[:, 0], probe_points_host[:, 1])

    abs_err = np.abs(u_num_host - u_ref_host)
    rel_l2 = np.linalg.norm(abs_err) / np.linalg.norm(u_ref_host)

    result = {
        'vol_nlevels': vol_nlevels,
        'nelements_bdry': nelements_bdry,
        'qbx_order': qbx_order,
        'fmm_order': fmm_order,
        'h': (bbox_b - bbox_a) / (2**vol_nlevels),
        'adaptive_boundary_max_level': adaptive_boundary_max_level,
        'adaptive_near_boundary_factor': float(adaptive_near_boundary_factor),
        'n_sources': int(source_points_host.shape[0]),
        'n_cells': int(mesh_info['n_active_cells']),
        'mesh_max_leaf_level': int(mesh_info['max_leaf_level']),
        'mesh_refine_passes': int(mesh_info['passes']),
        'mesh_refined_leaf_boxes': int(mesh_info['n_refined_total']),
        'n_source_interior': int(np.count_nonzero(interior_source_mask)),
        'n_source_exterior': int(np.count_nonzero(exterior_source_mask)),
        'n_probe': int(probe_points_host.shape[0]),
        'rel_l2_err': float(rel_l2),
        'max_abs_err': float(abs_err.max()),
        'p95_abs_err': float(np.percentile(abs_err, 95)),
        'gmres_rhs_ext_state': gmres_rhs_ext_state,
        'gmres_corr_state': dbg_corr['gmres_result'].state,
    }

    if return_fields:
        result.update(
            {
                'source_points_host': source_points_host,
                'source_vals_host': source_vals_host,
                'u_num_host': u_num_host,
                'u_ref_host': u_ref_host,
                'abs_err': abs_err,
                'bdry_points_host': bdry_points_host,
            }
        )

    if return_mesh:
        result.update(
            {
                'mesh_leaf_centers_host': np.ascontiguousarray(volume_mesh._leaf_centers()),
                'mesh_leaf_side_lengths_host': np.ascontiguousarray(volume_mesh._leaf_side_lengths()),
            }
        )

    return result


## 4) Single-Resolution Walkthrough

Run one representative configuration and inspect:

- geometry and sampling locations,
- interior/exterior source split,
- where harmonic continuation changes source values,
- numerical solution and interior error distribution.



In [ ]:
demo_result = run_poisson_case(
    vol_nlevels=4,
    nelements_bdry=192,
    qbx_order=4,
    fmm_order=14,
    return_fields=True,
)

n_src_total = demo_result['n_sources']
n_src_ext = demo_result['n_source_exterior']
ext_ratio = (n_src_ext / n_src_total) if n_src_total else 0.0

print('demo configuration:')
print(
    '  vol_nlevels=', demo_result['vol_nlevels'],
    'nelements_bdry=', demo_result['nelements_bdry'],
    'qbx_order=', demo_result['qbx_order'],
    'fmm_order=', demo_result['fmm_order'],
)
print('  n_sources=', demo_result['n_sources'], 'n_cells=', demo_result['n_cells'], 'n_probe=', demo_result['n_probe'])
print('  mesh max leaf level=', demo_result['mesh_max_leaf_level'], 'refine passes=', demo_result['mesh_refine_passes'])
print('  source split (in, out)=', demo_result['n_source_interior'], demo_result['n_source_exterior'])
print(f'  exterior-source fraction={ext_ratio:.3f}')
print('  gmres states (rhs-ext, corr)=', demo_result['gmres_rhs_ext_state'], demo_result['gmres_corr_state'])
print('  rel_l2_err=', demo_result['rel_l2_err'])
print('  max_abs_err=', demo_result['max_abs_err'])
print('  p95_abs_err=', demo_result['p95_abs_err'])



In [ ]:
import matplotlib.pyplot as plt

source_points = demo_result['source_points_host']
source_vals = demo_result['source_vals_host']
u_num = demo_result['u_num_host']
u_ref = demo_result['u_ref_host']
abs_err = demo_result['abs_err']

interior_src_mask = starfish_path.contains_points(source_points)
exterior_src_mask = ~interior_src_mask
rhs_true_on_sources = rhs_f(source_points[:, 0], source_points[:, 1])
rhs_ext_delta = source_vals - rhs_true_on_sources

_, probe_vertices_vis = make_starfish_path(scale=probe_scale, npts=4096)

fig, ax = plt.subplots(2, 3, figsize=(16, 9.5), constrained_layout=True)

ax00 = ax[0, 0]
ax00.plot(starfish_vertices[:, 0], starfish_vertices[:, 1], color='black', lw=1.6, label='domain boundary')
ax00.plot(probe_vertices_vis[:, 0], probe_vertices_vis[:, 1], color='tab:green', lw=1.1, ls='--', label=f'probe contour ({probe_scale:.2f}x)')
ax00.scatter(probe_points_host[:, 0], probe_points_host[:, 1], s=5, color='tab:blue', alpha=0.30, label='probe points')
ax00.set_title('Domain and Probe Set')
ax00.set_aspect('equal')
ax00.legend(loc='upper right', fontsize=8)

ax01 = ax[0, 1]
ax01.plot(starfish_vertices[:, 0], starfish_vertices[:, 1], color='black', lw=1.0)
ax01.scatter(source_points[interior_src_mask, 0], source_points[interior_src_mask, 1], s=5, alpha=0.35, color='tab:orange', label='interior sources')
ax01.scatter(source_points[exterior_src_mask, 0], source_points[exterior_src_mask, 1], s=5, alpha=0.35, color='tab:red', label='exterior sources (extended)')
ax01.set_title('Volume Source Point Split')
ax01.set_aspect('equal')
ax01.legend(loc='upper right', fontsize=8)

ax02 = ax[0, 2]
sc_rhs = ax02.scatter(source_points[:, 0], source_points[:, 1], c=rhs_ext_delta, s=8, cmap='coolwarm')
ax02.plot(starfish_vertices[:, 0], starfish_vertices[:, 1], color='black', lw=1.0)
ax02.set_title(r'RHS continuation delta: $f_{used} - f_{analytic}$')
ax02.set_aspect('equal')
cb_rhs = fig.colorbar(sc_rhs, ax=ax02, shrink=0.84)
cb_rhs.set_label('delta')

ax10 = ax[1, 0]
sc_u = ax10.scatter(probe_points_host[:, 0], probe_points_host[:, 1], c=u_num, s=10, cmap='viridis')
ax10.plot(starfish_vertices[:, 0], starfish_vertices[:, 1], color='black', lw=1.0)
ax10.set_title('Numerical Solution on Probe Set')
ax10.set_aspect('equal')
cb_u = fig.colorbar(sc_u, ax=ax10, shrink=0.84)
cb_u.set_label('u_num')

ax11 = ax[1, 1]
sc_e = ax11.scatter(probe_points_host[:, 0], probe_points_host[:, 1], c=np.log10(abs_err + 1.0e-14), s=10, cmap='magma')
ax11.plot(starfish_vertices[:, 0], starfish_vertices[:, 1], color='black', lw=1.0)
ax11.set_title(r'Probe error map: $\log_{10}(|u_{num}-u_{exact}|)$')
ax11.set_aspect('equal')
cb_e = fig.colorbar(sc_e, ax=ax11, shrink=0.84)
cb_e.set_label('log10 abs err')

ax12 = ax[1, 2]
ax12.hist(abs_err, bins=40, color='tab:purple', alpha=0.85)
ax12.axvline(np.percentile(abs_err, 50), color='tab:blue', ls='--', lw=1.5, label='p50')
ax12.axvline(np.percentile(abs_err, 95), color='tab:red', ls='--', lw=1.5, label='p95')
ax12.set_title('Probe Error Distribution')
ax12.set_xlabel('absolute error')
ax12.set_ylabel('count')
ax12.legend(loc='upper right', fontsize=8)

fig.suptitle(
    'Single-Resolution Diagnostics'
    + f"\nrel L2={demo_result['rel_l2_err']:.3e}, max={demo_result['max_abs_err']:.3e}, p95={demo_result['p95_abs_err']:.3e}",
    fontsize=12,
)

plt.show()



## 5) Uniform Co-Refinement Study

To estimate asymptotic behavior, refine together:

- volume mesh level (`vol_nlevels`),
- boundary element count (`nelements_bdry`),
- QBX order (`qbx_order`),
- FMM order (`fmm_order`, mildly increased).

This keeps one subsystem from becoming the dominant error floor too early.



In [ ]:
co_refinement_plan = [
    {'vol_nlevels': 3, 'nelements_bdry': 96, 'qbx_order': 3, 'fmm_order': 12},
    {'vol_nlevels': 4, 'nelements_bdry': 192, 'qbx_order': 4, 'fmm_order': 14},
    {'vol_nlevels': 5, 'nelements_bdry': 384, 'qbx_order': 5, 'fmm_order': 16},
    {'vol_nlevels': 6, 'nelements_bdry': 768, 'qbx_order': 6, 'fmm_order': 18},
]

co_results = []

print(f'Co-refinement (harmonic correction) on fixed interior probe set (starfish scaled by {probe_scale:.2f}):')
print(' level   n_bdry  qbx fmm      h      n_src   rel_l2_err   max_abs_err    p95_abs   rate  gmres')

prev_err = None
for cfg in co_refinement_plan:
    row = run_poisson_case(**cfg, return_fields=False)
    co_results.append(row)

    err = row['rel_l2_err']
    rate = np.log2(prev_err / err) if (prev_err is not None and err > 0.0) else np.nan
    rate_str = f'{rate:5.2f}' if np.isfinite(rate) else '  -  '

    gmres_ok = (row['gmres_rhs_ext_state'] == 'success' and row['gmres_corr_state'] == 'success')

    print(
        f"{row['vol_nlevels']:6d}"
        f" {row['nelements_bdry']:8d}"
        f" {row['qbx_order']:4d}"
        f" {row['fmm_order']:3d}"
        f" {row['h']:8.5f}"
        f" {row['n_sources']:8d}"
        f" {row['rel_l2_err']:12.4e}"
        f" {row['max_abs_err']:12.4e}"
        f" {row['p95_abs_err']:10.3e}"
        f" {rate_str:>5s}"
        f" {'ok' if gmres_ok else 'fail'}"
    )

    prev_err = err

hs = np.array([row['h'] for row in co_results])
errs = np.array([row['rel_l2_err'] for row in co_results])
max_errs = np.array([row['max_abs_err'] for row in co_results])
nsrc = np.array([row['n_sources'] for row in co_results])
levels = np.array([row['vol_nlevels'] for row in co_results])

if len(errs) >= 2 and np.all(errs > 0.0):
    slope, intercept = np.polyfit(np.log(hs), np.log(errs), 1)
    print(f'Harmonic-correction log-log slope (error vs h): {slope:.3f}')
else:
    slope = np.nan
    print('Not enough positive error data for slope estimate.')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

ax0 = axes[0]
ax0.loglog(hs, errs, 'o-', lw=2.0, ms=8, label='rel L2 error')
ax0.loglog(hs, max_errs, 's--', lw=1.6, ms=6, label='max abs error')
if np.isfinite(slope):
    fit = np.exp(intercept) * hs**slope
    ax0.loglog(hs, fit, ':', lw=1.6, label=f'fit slope={slope:.2f}')
ax0.set_xlabel('h (volume cell size scale)')
ax0.set_ylabel('error')
ax0.set_title('Error vs h (uniform co-refinement)')
ax0.grid(True, which='both', ls=':', alpha=0.5)
ax0.legend()

ax1 = axes[1]
ax1.loglog(nsrc, errs, 'o-', lw=2.0, ms=8, color='tab:blue')
for x, y, lev in zip(nsrc, errs, levels):
    ax1.annotate(f'l{lev}', (x, y), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax1.set_xlabel('number of volume source points')
ax1.set_ylabel('relative L2 error')
ax1.set_title('Accuracy-cost trend')
ax1.grid(True, which='both', ls=':', alpha=0.5)

plt.show()



## 6) Interpreting Convergence Results

Use the diagnostics below to answer practical questions:

- Do errors decrease consistently under co-refinement?
- Is GMRES robust across runs?
- What is the observed error-vs-resolution slope?
- Is the final accuracy good enough for your target workflow?



In [ ]:
best = min(co_results, key=lambda row: row['rel_l2_err'])
last = co_results[-1]
first = co_results[0]

overall_reduction = first['rel_l2_err'] / last['rel_l2_err'] if last['rel_l2_err'] > 0 else np.inf

print('Summary observations:')
print(f"- Best co-refined run: level={best['vol_nlevels']} with rel L2={best['rel_l2_err']:.4e}")
print(f"- Relative L2 reduction (first -> last): {overall_reduction:.2f}x")
print(f"- Final relative L2 error: {last['rel_l2_err']:.4e}")
print(f"- Final max abs error: {last['max_abs_err']:.4e}")

all_gmres_ok = all(
    row['gmres_rhs_ext_state'] == 'success' and row['gmres_corr_state'] == 'success'
    for row in co_results
)
print(f"- GMRES successful on all co-refinement runs: {all_gmres_ok}")

if np.isfinite(slope):
    if slope > 3.0:
        slope_note = 'strong high-order trend for this smooth manufactured case'
    elif slope > 1.0:
        slope_note = 'clear convergence trend'
    elif slope > 0.0:
        slope_note = 'convergent but moderate asymptotic rate'
    else:
        slope_note = 'non-convergent trend (investigate resolution balance)'
    print(f"- Fitted slope interpretation: {slope:.3f} -> {slope_note}")



## 7) Boundary-Focused AMR Study

Now compare a uniform level-5 mesh against boundary-focused adaptive refinement.

AMR marker used here:

- mark boxes with mixed inside/outside status on a 9-point stencil (corners + edge midpoints + center),
- use `adaptive_near_boundary_factor=0.0` so refinement follows geometric intersection markers,
- cap adaptive passes by case (`adaptive_max_passes`) to prioritize accuracy-per-cost.

Goal: improve boundary-region volume resolution without refining the whole box domain.

Error is measured on the full interior probe set by default (`probe_scale=1.00`), and below we also visualize the volume meshes to compare uniform vs AMR resolution patterns.


In [ ]:
amr_compare_plan = [
    {
        'label': 'uniform-l5',
        'vol_nlevels': 5,
        'nelements_bdry': 768,
        'qbx_order': 6,
        'fmm_order': 18,
        'adaptive_boundary_max_level': None,
        'adaptive_near_boundary_factor': 0.00,
        'adaptive_max_passes': 0,
    },
    {
        'label': 'amr-l5-to6',
        'vol_nlevels': 5,
        'nelements_bdry': 768,
        'qbx_order': 6,
        'fmm_order': 18,
        'adaptive_boundary_max_level': 6,
        'adaptive_near_boundary_factor': 0.00,
        'adaptive_max_passes': 1,
    },
    {
        'label': 'amr-l5-to7',
        'vol_nlevels': 5,
        'nelements_bdry': 768,
        'qbx_order': 6,
        'fmm_order': 18,
        'adaptive_boundary_max_level': 7,
        'adaptive_near_boundary_factor': 0.00,
        'adaptive_max_passes': 2,
    },
]

amr_results = []

print('Adaptive boundary-refinement comparison (manufactured-solution error):')
print('       case  n_cells   n_src  max_lev passes   rel_l2_err   max_abs_err    p95_abs  gmres')

for cfg in amr_compare_plan:
    try:
        row = run_poisson_case(
            vol_nlevels=cfg['vol_nlevels'],
            nelements_bdry=cfg['nelements_bdry'],
            qbx_order=cfg['qbx_order'],
            fmm_order=cfg['fmm_order'],
            adaptive_boundary_max_level=cfg['adaptive_boundary_max_level'],
            adaptive_max_passes=cfg['adaptive_max_passes'],
            adaptive_near_boundary_factor=cfg['adaptive_near_boundary_factor'],
            return_fields=False,
            return_mesh=True,
        )
    except Exception as exc:
        row = {
            'label': cfg['label'],
            'status': 'fail',
            'error': repr(exc),
        }
        amr_results.append(row)
        print(f"{cfg['label']:>11s}  FAIL  {row['error']}")
        continue

    row['label'] = cfg['label']
    row['status'] = 'ok'
    amr_results.append(row)

    gmres_ok = (row['gmres_rhs_ext_state'] == 'success' and row['gmres_corr_state'] == 'success')

    print(
        f"{row['label']:>11s}"
        f" {row['n_cells']:8d}"
        f" {row['n_sources']:7d}"
        f" {row['mesh_max_leaf_level']:8d}"
        f" {row['mesh_refine_passes']:6d}"
        f" {row['rel_l2_err']:12.4e}"
        f" {row['max_abs_err']:12.4e}"
        f" {row['p95_abs_err']:10.3e}"
        f" {'ok' if gmres_ok else 'fail'}"
    )

successful = {row['label']: row for row in amr_results if row.get('status') == 'ok'}
if 'uniform-l5' in successful and 'amr-l5-to7' in successful:
    uniform_row = successful['uniform-l5']
    amr_row = successful['amr-l5-to7']

    err_improvement = uniform_row['rel_l2_err'] / amr_row['rel_l2_err']
    src_growth = amr_row['n_sources'] / uniform_row['n_sources']

    efficiency_gain = err_improvement / src_growth if src_growth > 0 else np.nan

    print()
    print(f"Level-5 AMR improvement factor (rel L2, to level 7): {err_improvement:.3f}x")
    print(f"Source-point growth factor (to level 7): {src_growth:.3f}x")
    print(f"Accuracy-per-cost gain (improvement/growth): {efficiency_gain:.3f}")
else:
    print()
    print('AMR comparison factors unavailable because one or more runs failed.')

ok_rows = [row for row in amr_results if row.get('status') == 'ok']
if ok_rows:
    labels = [row['label'] for row in ok_rows]
    nsrc = np.array([row['n_sources'] for row in ok_rows], dtype=np.float64)
    rel = np.array([row['rel_l2_err'] for row in ok_rows], dtype=np.float64)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

    ax0 = axes[0]
    ax0.loglog(nsrc, rel, 'o-', lw=2.0, ms=8, color='tab:blue')
    for x, y, label in zip(nsrc, rel, labels):
        ax0.annotate(label, (x, y), textcoords='offset points', xytext=(4, 4), fontsize=8)
    ax0.set_xlabel('number of volume source points')
    ax0.set_ylabel('relative L2 error')
    ax0.set_title('AMR accuracy-cost trajectory')
    ax0.grid(True, which='both', ls=':', alpha=0.5)

    ax1 = axes[1]
    base_rel = rel[0]
    improvement = base_rel / rel
    ax1.bar(labels, improvement, color=['tab:gray', 'tab:orange', 'tab:green'][:len(labels)])
    ax1.set_ylabel('improvement factor vs uniform-l5')
    ax1.set_title('AMR relative improvement')
    ax1.grid(True, axis='y', ls=':', alpha=0.5)

    plt.show()

if 'uniform-l5' in successful and 'amr-l5-to7' in successful:
    from matplotlib.collections import LineCollection

    def box_segments_from_centers_sizes(centers, sizes):
        cx = np.asarray(centers[:, 0], dtype=np.float64)
        cy = np.asarray(centers[:, 1], dtype=np.float64)
        h = 0.5 * np.asarray(sizes, dtype=np.float64)
        x0 = cx - h
        x1 = cx + h
        y0 = cy - h
        y1 = cy + h

        segs = np.empty((4 * cx.size, 2, 2), dtype=np.float64)
        segs[0::4, 0, 0], segs[0::4, 0, 1] = x0, y0
        segs[0::4, 1, 0], segs[0::4, 1, 1] = x1, y0
        segs[1::4, 0, 0], segs[1::4, 0, 1] = x1, y0
        segs[1::4, 1, 0], segs[1::4, 1, 1] = x1, y1
        segs[2::4, 0, 0], segs[2::4, 0, 1] = x1, y1
        segs[2::4, 1, 0], segs[2::4, 1, 1] = x0, y1
        segs[3::4, 0, 0], segs[3::4, 0, 1] = x0, y1
        segs[3::4, 1, 0], segs[3::4, 1, 1] = x0, y0
        return segs

    mesh_rows = [successful['uniform-l5'], successful['amr-l5-to7']]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)

    for ax, row in zip(axes, mesh_rows):
        segs = box_segments_from_centers_sizes(
            row['mesh_leaf_centers_host'],
            row['mesh_leaf_side_lengths_host'],
        )
        lc = LineCollection(segs, colors='tab:blue', linewidths=0.22, alpha=0.45)
        ax.add_collection(lc)
        ax.plot(starfish_vertices[:, 0], starfish_vertices[:, 1], color='black', lw=1.3)
        ax.set_xlim(bbox_a, bbox_b)
        ax.set_ylim(bbox_a, bbox_b)
        ax.set_aspect('equal')
        ax.set_xlabel('x')
        ax.set_title(f"{row['label']}\n{row['n_cells']} cells, {row['n_sources']} sources")
        ax.grid(True, ls=':', alpha=0.35)

    axes[0].set_ylabel('y')
    fig.suptitle('Volume mesh comparison: uniform vs boundary-focused AMR', fontsize=12)
    plt.show()

